# 基于 OpenAI Python SDK 的模型接入方式

主要内容：在不依赖 LangChain 的前提下，如何使用 OpenAI Python SDK 接入不同来源的模型服务。

适用范围：
- 模型供应商提供的官方 API
- 模型云平台提供的 OpenAI 兼容端点
- 本地模型服务提供的 OpenAI 兼容接口

本文重点说明 3 个接入参数：
- `api_key`：身份认证
- `base_url`：请求目标服务地址
- `model`：具体调用的模型名称


## 一、依赖环境准备

安装依赖：

```bash
uv add openai
uv add python-dotenv
```

在项目根目录创建 `.env` 文件，按需配置不同平台的 API Key。


## 二、三类模型来源的接入思路

虽然模型来源不同，但通过 OpenAI Python SDK 接入时，核心配置通常仍然围绕以下 3 项展开：
- `api_key`：身份认证
- `base_url`：请求目标服务
- `model`：指定调用模型


下面的 3 个示例分别对应 3 种常见模型来源。

重点不是比较模型能力，而是观察：当模型来源变化时，`api_key`、`base_url`、`model` 这 3 项配置如何变化。


In [ ]:
# 1. 模型供应商：以 DeepSeek 官方 API 为例
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

client = OpenAI(
    api_key=os.environ.get('DEEPSEEK_API_KEY'),
    base_url="https://api.deepseek.com"
)

response = client.chat.completions.create(
    model="deepseek-v4-flash",
    messages=[
        {"role":"system", "content":"你是一个专业的辅助学习LangChain的助手！"},
        {"role":"user", "content":"简要告诉我，LangChain能做什么？"}
    ],
    stream=False,
    reasoning_effort="low"
)

print(response.choices[0].message.content)

In [ ]:
# 2. 模型云平台：以阿里云百炼为例

import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

# 注意: 不同地域的base_url不通用（下方示例使用北京地域的 base_url）
# - 华北2（北京）: https://dashscope.aliyuncs.com/compatible-mode/v1
# - 美国（弗吉尼亚）: https://dashscope-us.aliyuncs.com/compatible-mode/v1
# - 新加坡: https://dashscope-intl.aliyuncs.com/compatible-mode/v1
# - 德国（法兰克福）: https://{WorkspaceId}.eu-central-1.maas.aliyuncs.com/compatible-mode/v1，请将WorkspaceId替换为业务空间ID

client = OpenAI(
    api_key=os.environ.get('DASHSCOPE_API_KEY'),
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
)
completion = client.chat.completions.create(
    # model="qwen3.5-122b-a10b",
    # model="deepseek-v4-flash",
    # model="glm-5",
    model="kimi-k2.6",
    messages=[{'role': 'user', 'content': '你是谁？'}]
)
print(completion.choices[0].message.content)

In [ ]:
# 3. 本地模型服务：以 Ollama 为例
# Ollama 官方明确支持 OpenAI compatibility

from openai import OpenAI

client = OpenAI(
    base_url='http://localhost:11434/v1/',
    api_key='ollama',  # required but ignored
)

chat_completion = client.chat.completions.create(
    messages=[
        {
            'role': 'user',
            'content': '你是谁',
        }
    ],
    # model='qwen2.5:32b',
    model='gemma4:31b',
)
print(chat_completion.choices[0].message.content)


## 三、简要总结

这 3 种方式在调用形式上基本一致，核心区别主要体现在接入参数：
- 官方 Provider：通常使用官方 API 地址
- 模型云平台：通常使用平台提供的 OpenAI 兼容端点
- 本地模型服务：通常使用本地 `base_url`，有时 `api_key` 只是占位值

如果目标服务兼容 OpenAI API，通常就可以沿用同一套 OpenAI Python SDK 调用方式。
